In [1]:
import sympy as sym
from microscope_calibration.common.model import Model4DSTEM, Ray, PixelYX, DescanError
from numpy.testing import assert_allclose
import jax

In [2]:
def equals(ray1: Ray, ray2: Ray) -> bool:
    for key in ray1.__dict__.keys():
        if not sym.sympify(ray1.__dict__[key]).equals(ray2.__dict__[key]):
            return False
    return True

In [3]:
sym.sympify(0).equals(0.0)

True

In [4]:
x = sym.Symbol('x')

In [5]:
sym.sympify(x)

x

In [6]:
sym.sympify(0.0) == 0

False

In [7]:
px = PixelYX(0., 0.)
px.__annotations__

{'y': typing.Union[int, float], 'x': typing.Union[int, float]}

In [8]:
px._asdict()

{'y': 0.0, 'x': 0.0}

In [9]:
ray = Ray(0,0,0,0,0,0)

In [10]:
px.x

0.0

In [11]:
def test_scan(dy, dx, scan_y, scan_x):
    model = Model4DSTEM(
        overfocus=1,
        scan_pixel_pitch=1,
        scan_center=PixelYX(y=0., x=0.),
        scan_rotation=0.,
        camera_length=1,
        detector_pixel_pitch=1,
        detector_center=PixelYX(y=0., x=0.),
        semiconv=0.023,
        flip_factor=1.,
        descan_error=DescanError()
    )
    res_straight = model.trace(scan_pos=PixelYX(y=0., x=0.), source_dx=dx, source_dy=dy)
    res = model.trace(scan_pos=PixelYX(y=scan_y, x=scan_x), source_dx=dx, source_dy=dy)

    for key in res.keys():
        sect = res[key]
        sect_straight = res_straight[key]
        assert sect.ray.z == sect_straight.ray.z
        assert sect.ray.pathlength == sect_straight.ray.pathlength
        # if isinstance(sect.component, Component) or isinstance(sect.component, Source):
            # assert sect.component.z == sect.ray.z
            # assert sect.component.z == sect.ray.pathlength
        # Beam is deflected
        if key in ('scanner', 'specimen'):
            print(type(sect.ray.x), type(sect_straight.ray.x), sect.ray.x, sect_straight.ray.x)
            assert sym.sympify(sect.ray.x - sect_straight.ray.x).equals(scan_x)
            assert sect.ray.y - sect_straight.ray.y == scan_y
        # Beam is not deflected
        else:
            assert_allclose(sect.ray.x, sect_straight.ray.x)
            assert_allclose(sect.ray.y, sect_straight.ray.y)
            # Ray propagates straight
            assert_allclose(sect.ray.x, sect.ray.z * dx)
            assert_allclose(sect.ray.y, sect.ray.z * dy)
    assert_allclose(res['detector'].ray.z, model.overfocus + model.camera_length)
    assert_allclose(res['source'].ray.z, 0.)
    # Correct scan deflection
    assert_allclose(
        res['specimen'].sampling['scan_px'],
        PixelYX(
            x=scan_x + res_straight['specimen'].sampling['scan_px'].x,
            y=scan_y + res_straight['specimen'].sampling['scan_px'].y
        )
    )
    # Check that central ray goes through scan position
    if dx == 0. and dy == 0.:
        assert_allclose(
            res['specimen'].sampling['scan_px'],
            PixelYX(
                x=scan_x,
                y=scan_y,
            ),
            rtol=1e-12, atol=1e-12
        )
    # check physical coords equals pixel coords
    assert_allclose(
        res['specimen'].sampling['scan_px'],
        PixelYX(
            x=res['specimen'].ray.x,
            y=res['specimen'].ray.y,
        )
    )
    assert_allclose(res['detector'].sampling['detector_px'], PixelYX(
        x=dx*(model.overfocus + model.camera_length),
        y=dy*(model.overfocus + model.camera_length)
    ))
    # check physical coords equals pixel coords
    assert_allclose(
        res['detector'].sampling['detector_px'],
        PixelYX(
            x=res['detector'].ray.x,
            y=res['detector'].ray.y,
        )
    )

In [12]:
#test_scan(-0.2, -0.7, -17, -23)

In [13]:
sym.Float(0.34) - sym.Float(0.34) == sym.Float(0.0)

False

In [14]:
from copy import deepcopy
from typing import TypeVar, Callable
from typing import Optional, NamedTuple, Union, Any, TypeVar, get_args
from collections.abc import Container
from numbers import Number

In [75]:
T = TypeVar("T")

def isnumerictype(T):
    for supercls in (Number, sym.Number):
        # try ... except because some reasonable Ts like int | float cause a TypeError here
        try:
            if issubclass(T, supercls):
                return True
        except TypeError:
            pass
    # This can also handle unions like int | float via isinstance()
    for number in (0, 0., 0.+0.j):
        # try ... except because weird types can cause a TypeError here,
        # but we can safely continue since we didn't positively recognize it as numeric
        try:
            if isinstance(number, T):
                return True
        except TypeError:
            pass
    return False

def symbol_maker(
        params_cls: type[T], postfix: str | None = None,
        recurse_for: Container = tuple()) -> T:
    """
    Declare sympy symbols for each attribute of a given parameter class.

    Use __annotations__ to access the class attributes and their types.
    Make a dictionary of sympy symbols assigned to each class attribute of
    a primitive type. The symbol names are declared as names of the attributes
    with an appropriate postfix. After that construct a class instance with
    the symbolic variables as parameters.

    Parameters
    ----------
    params_cls: class
        class which needs symbolic variables to be assigned to its parameters
    postfix: str
        postfix for each symbol name to specify the parameters (e.g. 'new' or 'old')
    recurse_for: tuple
        list of class attributes of non-primitive types to receive symbols for each component

    Returns
    -------
    class instance
        instance of the class with symbols as parameters
    """
    def symbol_maker_inner(params_cls: type[T], postfix: str | None,
            recurse_for: Container, index: int) -> tuple[int, T]:
        if hasattr(params_cls, '__annotations__'):
            symbols_dict = {}
            for attr in params_cls.__annotations__.keys():
                cls = params_cls.__annotations__[attr]
                if cls in recurse_for:
                    (index, symbols_dict[attr]) = symbol_maker_inner(
                        cls, postfix, recurse_for, index
                    )
                elif isnumerictype(cls):
                    sym_name = attr if postfix is None else f"{attr}_{postfix}"
                    sym_name = f"{sym_name}_{index}"
                    symbols_dict[attr] = sym.Symbol(sym_name)
                    index += 1
                else:
                    raise TypeError(f"Can't generate symbol for type {cls}.")
            return (index, params_cls(**symbols_dict))
        elif isnumerictype(params_cls):
            attr = params_cls.__name__
            sym_name = attr if postfix is None else f"{attr}_{postfix}"
            sym_name = f"{sym_name}_{index}"
            index += 1
            return (index, sym.Symbol(sym_name))
        else:
            raise TypeError(f"Can't generate symbol for type {params_cls}.")

    (_, res) = symbol_maker_inner(
        params_cls=params_cls,
        postfix=postfix,
        recurse_for=recurse_for,
        index=0
    )
    return res


SymbolJaxTree = TypeVar("SymbolJaxTree")


def lambdify(inp: SymbolJaxTree, outp: SymbolJaxTree, **kwargs):
    inp_leaves, inp_treedef = jax.tree.flatten(inp)
    outp_leaves, outp_treedef = jax.tree.flatten(outp)

    inp_indices = []
    inp_symbols = []
    inp_dups = {}

    for i, leave in enumerate(inp_leaves):
        if isinstance(leave, sym.Symbol):
            if leave in inp_symbols:
                inp_dups[i] = inp_symbols.index(leave)
            else:
                inp_indices.append(i)
                inp_symbols.append(leave)
        elif is_sympy(leave) and not isinstance(leave, (sym.NumberSymbol, sym.Number)):
            raise ValueError(
                f"SymPy leave {leave} found that is not a symbol or a constant number. "
                "Only symbols and constants are allowed in the input definition."
            )

    outp_indices = []
    outp_exprs = []

    for i, leave in enumerate(outp_leaves):
        if isinstance(leave, sym.Basic):
            outp_indices.append(i)
            outp_exprs.append(leave)

    inp_indices_set = set(inp_indices)
    inner_f = sym.lambdify(inp_symbols, outp_exprs, **kwargs)

    def outer(ii):
        ii_leaves, ii_treedef = jax.tree.flatten_with_path(ii)
        if ii_treedef != inp_treedef:
            raise ValueError(
                f'Tree definition of input {ii_treedef} does not match expected '
                f'tree definition {inp_treedef}.'
            )
        for i, (path, leave) in enumerate(ii_leaves):
            if i not in inp_indices_set:
                if i in inp_dups:
                    orig_i = inp_dups[i]
                    orig_path, orig_leave = ii_leaves[orig_i]
                    if orig_leave != leave:
                        raise ValueError(
                            f"Input value {leave} with path {path} was a duplicate symbol in "
                            "original input "
                            f"but is now not matching the input value {orig_leave} at {orig_path}"
                        )
                elif leave != inp_leaves[i]:
                    raise ValueError(
                        f"Constant value {leave} doesn't match reference input "
                        f"{inp_leaves[i]} for {path}.")

        ii_vals = [ii_leaves[i][1] for i in inp_indices]
        oo_inner = inner_f(*ii_vals)
        outp = deepcopy(outp_leaves)
        for i, val in enumerate(oo_inner):
            index = outp_indices[i]
            outp[index] = val
        return jax.tree.unflatten(outp_treedef, outp)

    return outer

In [142]:
model = symbol_maker(params_cls=Model4DSTEM, postfix=None, recurse_for=[DescanError, PixelYX])
model

Model4DSTEM(overfocus=overfocus_0, scan_pixel_pitch=scan_pixel_pitch_1, scan_center=PixelYX(y=y_2, x=x_3), scan_rotation=scan_rotation_4, camera_length=camera_length_5, detector_pixel_pitch=detector_pixel_pitch_6, detector_center=PixelYX(y=y_7, x=x_8), semiconv=semiconv_9, flip_factor=flip_factor_10, descan_error=DescanError(pxo_pxi=pxo_pxi_11, pxo_pyi=pxo_pyi_12, pyo_pxi=pyo_pxi_13, pyo_pyi=pyo_pyi_14, sxo_pxi=sxo_pxi_15, sxo_pyi=sxo_pyi_16, syo_pxi=syo_pxi_17, syo_pyi=syo_pyi_18, offpxi=offpxi_19, offpyi=offpyi_20, offsxi=offsxi_21, offsyi=offsyi_22), detector_rotation=detector_rotation_23)

In [79]:
scan_pos = symbol_maker(PixelYX, postfix=None)
scan_pos

PixelYX(y=y_0, x=x_1)

In [18]:
isinstance(Number, type)

True

In [19]:
Number

numbers.Number

In [20]:
issubclass(Union[int, float], Number)

TypeError: issubclass() arg 1 must be a class

In [21]:
T = Union[int, float]

In [22]:
get_args(T)

(int, float)

In [23]:
type(T)

typing._UnionGenericAlias

In [24]:
model = Model4DSTEM(overfocus=sym.Symbol('overfocus_0'), 
                    scan_pixel_pitch=sym.Symbol('scan_pixel_pitch_1'),
                    scan_center=PixelYX(y=sym.Symbol('y_2'), x=sym.Symbol('x_3')),
                    scan_rotation=sym.Symbol('scan_rotation_4'),
                    camera_length=sym.Symbol('camera_length_5'),
                    detector_pixel_pitch=sym.Symbol('detector_pixel_pitch_6'),
                    detector_center=PixelYX(y=sym.Symbol('y_7'), x=sym.Symbol('x_8')),
                    semiconv=sym.Symbol('semiconv_9'),
                    flip_factor=sym.Symbol('flip_factor_10'),
                    descan_error=DescanError(pxo_pxi=sym.Symbol('pxo_pxi_11'), pxo_pyi=sym.Symbol('pxo_pyi_12'), pyo_pxi=sym.Symbol('pyo_pxi_13'), pyo_pyi=sym.Symbol('pyo_pyi_14'),
                         sxo_pxi=sym.Symbol('sxo_pxi_15'), sxo_pyi=sym.Symbol('sxo_pyi_16'), syo_pxi=sym.Symbol('syo_pxi_17'), syo_pyi=sym.Symbol('syo_pyi_18'),
                         offpxi=sym.Symbol('offpxi_19'), offpyi=sym.Symbol('offpyi_20'), offsxi=sym.Symbol('offsxi_21'), offsyi=sym.Symbol('offsyi_22')),
                    detector_rotation=sym.Symbol('detector_rotation_23'))

In [25]:
scan_pos = PixelYX(y=sym.Symbol('y_0'), x=sym.Symbol('x_0'))

In [26]:
model.trace(scan_pos=scan_pos, source_dx=sym.Symbol('source_dx'), source_dy=sym.Symbol('source_dy'))

OrderedDict([('source',
              ResultSection(component=PointSource(z=0, semi_conv=semiconv_9, offset_xy=CoordsXY(x=0.0, y=0.0)), ray=Ray(x=0, y=0, dx=source_dx, dy=source_dy, z=0, pathlength=0.0, _one=1.0), sampling=None)),
             ('overfocus',
              ResultSection(component=Propagator(distance=overfocus_0, propagator=<temgym_core.propagator.FreeSpaceParaxial object at 0x7fd59a377e00>), ray=Ray(x=overfocus_0*source_dx, y=overfocus_0*source_dy, dx=source_dx, dy=source_dy, z=overfocus_0, pathlength=overfocus_0, _one=1.0), sampling=None)),
             ('scanner',
              ResultSection(component=Scanner(z=overfocus_0, scan_pos_x=scan_pixel_pitch_1*(x_0 - 1.0*x_3)*cos(scan_rotation_4) - scan_pixel_pitch_1*(y_0 - 1.0*y_2)*sin(scan_rotation_4), scan_pos_y=scan_pixel_pitch_1*(x_0 - 1.0*x_3)*sin(scan_rotation_4) + scan_pixel_pitch_1*(y_0 - 1.0*y_2)*cos(scan_rotation_4), scan_tilt_x=0.0, scan_tilt_y=0.0), ray=Ray(x=overfocus_0*source_dx + 1.0*scan_pixel_pitch_1*(x_0 - 

In [27]:
def trace_lambdify(model, scan_pos, source_dx, source_dy):
    res = model.trace(scan_pos=scan_pos, source_dx=source_dx, source_dy=source_dy)
    f = lambdify((model, scan_pos, source_dx, source_dy), res)
    return f

In [28]:
trace_lambdify(model=model, scan_pos=scan_pos, source_dx=sym.Symbol('source_dx'), source_dy=sym.Symbol('source_dy'))

<function __main__.lambdify.<locals>.outer(ii)>

In [29]:
model_num = Model4DSTEM(
        overfocus=1,
        scan_pixel_pitch=1,
        scan_center=PixelYX(y=0., x=0.),
        scan_rotation=0.,
        camera_length=1,
        detector_pixel_pitch=1,
        detector_center=PixelYX(y=0., x=0.),
        semiconv=0.023,
        flip_factor=1.,
        descan_error=DescanError()
    )

In [30]:
scan_pos_num = PixelYX(0., 0.)

In [31]:
trace_lambdify(model=model, scan_pos=scan_pos, source_dx=sym.Symbol('source_dx'), source_dy=sym.Symbol('source_dy'))((model_num, scan_pos_num, 0., 0.))

OrderedDict([('source',
              ResultSection(component=PointSource(z=0, semi_conv=0.023, offset_xy=CoordsXY(x=0.0, y=0.0)), ray=Ray(x=0, y=0, dx=0.0, dy=0.0, z=0, pathlength=0.0, _one=1.0), sampling=None)),
             ('overfocus',
              ResultSection(component=Propagator(distance=1, propagator=<temgym_core.propagator.FreeSpaceParaxial object at 0x7fd583dfd810>), ray=Ray(x=0.0, y=0.0, dx=0.0, dy=0.0, z=1, pathlength=1, _one=1.0), sampling=None)),
             ('scanner',
              ResultSection(component=Scanner(z=1, scan_pos_x=np.float64(0.0), scan_pos_y=np.float64(0.0), scan_tilt_x=0.0, scan_tilt_y=0.0), ray=Ray(x=np.float64(0.0), y=np.float64(0.0), dx=0.0, dy=0.0, z=1, pathlength=1, _one=1.0), sampling=None)),
             ('specimen',
              ResultSection(component=Plane(z=1), ray=Ray(x=np.float64(0.0), y=np.float64(0.0), dx=0.0, dy=0.0, z=1, pathlength=1, _one=1.0), sampling={'scan_px': PixelYX(y=np.float64(0.0), x=np.float64(0.0))})),
             ('de

In [149]:
from typing import Union
model.trace(scan_pos=scan_pos, source_dx=0., source_dy=0.)

OrderedDict([('source',
              ResultSection(component=PointSource(z=0, semi_conv=semiconv_9, offset_xy=CoordsXY(x=0.0, y=0.0)), ray=Ray(x=0.0, y=0.0, dx=0.0, dy=0.0, z=0, pathlength=0.0, _one=1.0), sampling=None)),
             ('overfocus',
              ResultSection(component=Propagator(distance=overfocus_0, propagator=<temgym_core.propagator.FreeSpaceParaxial object at 0x7fd59a377e00>), ray=Ray(x=0, y=0, dx=0.0, dy=0.0, z=overfocus_0, pathlength=overfocus_0, _one=1.0), sampling=None)),
             ('scanner',
              ResultSection(component=Scanner(z=overfocus_0, scan_pos_x=scan_pixel_pitch_1*(x_1 - 1.0*x_3)*cos(scan_rotation_4) - scan_pixel_pitch_1*(y_0 - 1.0*y_2)*sin(scan_rotation_4), scan_pos_y=scan_pixel_pitch_1*(x_1 - 1.0*x_3)*sin(scan_rotation_4) + scan_pixel_pitch_1*(y_0 - 1.0*y_2)*cos(scan_rotation_4), scan_tilt_x=0.0, scan_tilt_y=0.0), ray=Ray(x=1.0*scan_pixel_pitch_1*(x_1 - 1.0*x_3)*cos(scan_rotation_4) - 1.0*scan_pixel_pitch_1*(y_0 - 1.0*y_2)*sin(scan_rota

In [33]:
T = Union[int, float]

In [44]:
isinstance(23., float)

True

In [39]:
issubclass(T.__class__, Number)

False

In [55]:
isinstance(sym.sympify(0.), sym.Number)

True

In [59]:
issubclass(sym.Float, sym.Number)

True

In [73]:
class Test(NamedTuple):
    a: 23

In [74]:
symbol_maker(Test)

23


TypeError: Can't generate symbol for type 23.

In [80]:
import numpy as np

In [81]:
x, y = sym.symbols('x y')

In [83]:
sym.sympify(np.array([x,y])).equals(np.array([x,y]))

AttributeError: 'ImmutableDenseNDimArray' object has no attribute 'equals'

In [86]:
x.equals(x) is True

True

In [116]:
x = sym.Symbol('x', nonzero=True)
y = sym.Symbol('y', nonzero=True)

In [117]:
if x != 0 or y != 0:
    print('a')
else:
    print('b')

a


In [118]:
if x.equals(0) is False or x.equals(0) is False:
    print('a')
else:
    print('b')

a


In [143]:
    class ReturnT(NamedTuple):
            phys_y: float
            phys_x: float
            pass_y: float
            pass_x: float
            detector_scan_px_y: float
            detector_scan_px_x: float

    def calculate(scan_y, scan_x):
        res = model.trace(
            scan_pos=PixelYX(x=scan_x, y=scan_y),
            source_dy=0.,
            source_dx=0.,
        )
        return ReturnT(
            phys_y=res['detector'].ray.y,
            phys_x=res['detector'].ray.x,
            pass_y=res['specimen'].ray.y,
            pass_x=res['specimen'].ray.x,
            detector_scan_px_y=res['detector'].sampling['detector_px'],
            detector_scan_px_x=res['specimen'].sampling['scan_px']
        )
    scan_y, scan_x = sym.symbols('scan_y scan_x')

    lambdifyed_res = lambdify(
        [scan_y, scan_x],
        [*calculate(scan_y, scan_x)]
    )

In [146]:
calculate(1,1).phys_y

camera_length_5*(1.0*offsyi_22 + 1.0*syo_pxi_17*(scan_pixel_pitch_1*(1 - 1.0*x_3)*cos(scan_rotation_4) - scan_pixel_pitch_1*(1 - 1.0*y_2)*sin(scan_rotation_4)) + 1.0*syo_pyi_18*(scan_pixel_pitch_1*(1 - 1.0*x_3)*sin(scan_rotation_4) + scan_pixel_pitch_1*(1 - 1.0*y_2)*cos(scan_rotation_4))) + 1.0*offpyi_20 + 1.0*pyo_pxi_13*(scan_pixel_pitch_1*(1 - 1.0*x_3)*cos(scan_rotation_4) - scan_pixel_pitch_1*(1 - 1.0*y_2)*sin(scan_rotation_4)) + 1.0*pyo_pyi_14*(scan_pixel_pitch_1*(1 - 1.0*x_3)*sin(scan_rotation_4) + scan_pixel_pitch_1*(1 - 1.0*y_2)*cos(scan_rotation_4))

In [127]:
de = DescanError()
de

DescanError(pxo_pxi=0.0, pxo_pyi=0.0, pyo_pxi=0.0, pyo_pyi=0.0, sxo_pxi=0.0, sxo_pyi=0.0, syo_pxi=0.0, syo_pyi=0.0, offpxi=0.0, offpyi=0.0, offsxi=0.0, offsyi=0.0)

In [135]:
[*model.descan_error]

[pxo_pxi_11,
 pxo_pyi_12,
 pyo_pxi_13,
 pyo_pyi_14,
 sxo_pxi_15,
 sxo_pyi_16,
 syo_pxi_17,
 syo_pyi_18,
 offpxi_19,
 offpyi_20,
 offsxi_21,
 offsyi_22]

In [134]:
sym.lambdify([*model.descan_error], model.descan_error.pxo_pxi+1)

<function _lambdifygenerated(pxo_pxi_11, pxo_pyi_12, pyo_pxi_13, pyo_pyi_14, sxo_pxi_15, sxo_pyi_16, syo_pxi_17, syo_pyi_18, offpxi_19, offpyi_20, offsxi_21, offsyi_22)>